In [ ]:
import torch
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
from lightning.pytorch.callbacks import (
  EarlyStopping,
  LearningRateMonitor,
  ModelCheckpoint,
  RichProgressBar,
)
from lightning.pytorch.loggers import CSVLogger
from pathlib import Path
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [ ]:
DATA_DIR = Path('../data')

In [ ]:
train = torch.load(f'{DATA_DIR}/bin/training_temporaldataset.pkl', weights_only=False)
val = torch.load(f'{DATA_DIR}/bin/validation_temporaldataset.pkl', weights_only=False)
test = torch.load(f'{DATA_DIR}/bin/test_temporaldataset.pkl', weights_only=False)

In [ ]:
# ==================== HYPERPARAMETERS ====================
# Model Architecture
HIDDEN_SIZE = 32  # Main model width (16-64 rec for crypto)
ATTENTION_HEAD_SIZE = 4  # Number of attention heads
HIDDEN_CONTINUOUS_SIZE = 16  # Size for continuous variable processing
LSTM_LAYERS = 2  # Number of LSTM layers in encoder/decoder
DROPOUT = 0.2  # Dropout rate (higher for noisy data like crypto)

# Training
LEARNING_RATE = 1e-3  # Initial learning rate
WEIGHT_DECAY = 1e-2  # L2 regularization for AdamW (0.01 standard for transformers)
BATCH_SIZE = 512  # Batch size (smaller for volatile data)
MAX_EPOCHS = 100  # Maximum training epochs
GRADIENT_CLIP_VAL = 0.1  # Gradient clipping threshold

# Early Stopping
PATIENCE = 15  # Epochs without improvement before stopping
MIN_DELTA = 1e-4  # Minimum change to qualify as improvement

# DataLoader
NUM_WORKERS = 0  # DataLoader workers (0 for notebook stability)

# Loss Function (QuantileLoss for uncertainty estimation)
QUANTILES = [0.1, 0.5, 0.9]  # 10th, 50th (median), 90th percentiles

print(f'Hyperparameters:')
print(
  f'  Model: hidden={HIDDEN_SIZE}, heads={ATTENTION_HEAD_SIZE}, lstm_layers={LSTM_LAYERS}, dropout={DROPOUT}'
)
print(
  f'  Training: lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY}, batch={BATCH_SIZE}, epochs={MAX_EPOCHS}'
)
print(f'  Early stopping: patience={PATIENCE}, min_delta={MIN_DELTA}')

In [ ]:
# Create DataLoaders
train_dataloader = train.to_dataloader(
  train=True, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
val_dataloader = val.to_dataloader(
  train=False,
  batch_size=BATCH_SIZE * 2,  # Larger batch for validation (faster)
  num_workers=NUM_WORKERS,
)

print(f'Train batches: {len(train_dataloader):,}')
print(f'Val batches: {len(val_dataloader):,}')

In [ ]:
# Initialize TFT Model
tft = TemporalFusionTransformer.from_dataset(
  train,
  hidden_size=HIDDEN_SIZE,
  attention_head_size=ATTENTION_HEAD_SIZE,
  hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE,
  lstm_layers=LSTM_LAYERS,
  dropout=DROPOUT,
  learning_rate=LEARNING_RATE,
  loss=QuantileLoss(quantiles=QUANTILES),
  optimizer='adamw',
  weight_decay=WEIGHT_DECAY,
  reduce_on_plateau_patience=4,
)

print(f'Model parameters: {tft.size() / 1e6:.2f}M')
print(f'Optimizer: AdamW with weight_decay={WEIGHT_DECAY}')
print(f'Loss function: QuantileLoss with quantiles {QUANTILES}')

In [ ]:
# Setup Trainer with Callbacks
early_stop = EarlyStopping(
  monitor='val_loss', patience=PATIENCE, min_delta=MIN_DELTA, mode='min', verbose=True
)

lr_monitor = LearningRateMonitor(
  logging_interval='step'
)  # Changed to 'step' for more frequent updates

checkpoint = ModelCheckpoint(
  monitor='val_loss',
  mode='min',
  save_top_k=1,
  filename='tft-{epoch:02d}-{val_loss:.4f}',
)


# Custom RichProgressBar to show learning rate
class CustomProgressBar(RichProgressBar):
  def get_metrics(self, trainer, model):
    items = super().get_metrics(trainer, model)
    # Add learning rate to displayed metrics
    if trainer.optimizers:
      items['lr'] = f'{trainer.optimizers[0].param_groups[0]["lr"]:.2e}'
    return items


# Set console width via console_kwargs to prevent wrapping
progress_bar = CustomProgressBar(console_kwargs={'width': 250})

# CSV Logger (fixes tensorboard warning)
logger = CSVLogger('lightning_logs', name='tft_training')

trainer = pl.Trainer(
  max_epochs=MAX_EPOCHS,
  gradient_clip_val=GRADIENT_CLIP_VAL,
  callbacks=[early_stop, lr_monitor, checkpoint, progress_bar],
  logger=logger,
  log_every_n_steps=50,
)

print(
  f'Trainer configured with max_epochs={MAX_EPOCHS}, gradient_clip={GRADIENT_CLIP_VAL}'
)
print(f'Logs will be saved to: {logger.log_dir}')

# Training

**Note**: Training will use MPS (Apple Silicon GPU) if available. If you encounter errors, you can disable GPU by changing `accelerator="cpu"` in the Trainer.


In [ ]:
# Train Model
trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

print(f'\nTraining complete!')
print(f'Best model saved at: {checkpoint.best_model_path}')

In [ ]:
# Visualize Training Progress
import pandas as pd

# Read training logs
metrics_path = f'{logger.log_dir}/metrics.csv'
metrics = pd.read_csv(metrics_path)

# Extract train and val loss per epoch
train_metrics = metrics[metrics['train_loss_epoch'].notna()][
  ['epoch', 'train_loss_epoch']
].rename(columns={'train_loss_epoch': 'train_loss'})
val_metrics = metrics[metrics['val_loss'].notna()][['epoch', 'val_loss']]

# Merge on epoch
loss_df = train_metrics.merge(val_metrics, on='epoch', how='outer').sort_values('epoch')

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
  loss_df['epoch'],
  loss_df['train_loss'],
  label='Train Loss',
  linewidth=2,
  marker='o',
  markersize=4,
)
ax.plot(
  loss_df['epoch'],
  loss_df['val_loss'],
  label='Val Loss',
  linewidth=2,
  marker='s',
  markersize=4,
)

# Mark best epoch
best_epoch = (
  checkpoint.best_model_path.split('epoch=')[1].split('-')[0]
  if checkpoint.best_model_path
  else None
)
if best_epoch:
  best_epoch = int(best_epoch)
  ax.axvline(
    best_epoch,
    color='red',
    linestyle='--',
    alpha=0.7,
    label=f'Best Model (epoch {best_epoch})',
  )

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (QuantileLoss)')
ax.set_title('Training vs Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Final train loss: {loss_df["train_loss"].iloc[-1]:.6f}')
print(f'Final val loss: {loss_df["val_loss"].iloc[-1]:.6f}')
print(
  f'Best val loss: {loss_df["val_loss"].min():.6f} at epoch {loss_df.loc[loss_df["val_loss"].idxmin(), "epoch"]:.0f}'
)

In [ ]:
# Load Best Model
best_model_path = checkpoint.best_model_path
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

print(f'Loaded best model from: {best_model_path}')

# Testing


In [ ]:
# Create Test DataLoader
test_dataloader = test.to_dataloader(
  train=False, batch_size=BATCH_SIZE * 4, num_workers=NUM_WORKERS
)

print(f'Test batches: {len(test_dataloader):,}')

In [ ]:
# Generate Predictions
predictions = best_tft.predict(
  test_dataloader, return_y=True, trainer_kwargs=dict(accelerator='cpu')
)

# predictions is a tuple: (predictions_tensor, actuals_tensor)
# For QuantileLoss: predictions has shape [n_samples, n_quantiles]
pred_tensor = predictions.output
actuals = predictions.y[0]  # Shape: [n_samples, 1]

print(f'Predictions shape: {pred_tensor.shape}')
print(f'Actuals shape: {actuals.shape}')

In [ ]:
# Calculate Metrics
# Extract quantile predictions: [q10, q50 (median), q90]
pred_np = pred_tensor.cpu().numpy()
actual_np = actuals.cpu().numpy().flatten()

# Use median (q50) for point prediction metrics
median_idx = QUANTILES.index(0.5)
pred_median = pred_np[:, median_idx]

# Point prediction metrics
mae = mean_absolute_error(actual_np, pred_median)
rmse = np.sqrt(mean_squared_error(actual_np, pred_median))
r2 = r2_score(actual_np, pred_median)

# Coverage: % of actuals within prediction interval [q10, q90]
q10_idx = QUANTILES.index(0.1)
q90_idx = QUANTILES.index(0.9)
pred_q10 = pred_np[:, q10_idx]
pred_q90 = pred_np[:, q90_idx]
coverage = np.mean((actual_np >= pred_q10) & (actual_np <= pred_q90)) * 100

print('=' * 50)
print('TEST SET METRICS')
print('=' * 50)
print(f'MAE:      {mae:.6f}')
print(f'RMSE:     {rmse:.6f}')
print(f'R²:       {r2:.4f}')
print(f'Coverage: {coverage:.1f}% (target: 80% for [0.1, 0.9] interval)')
print('=' * 50)

In [ ]:
# Visualization: Sample predictions with uncertainty bands
n_samples = 200  # Show first N predictions

fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(n_samples)
ax.plot(x, actual_np[:n_samples], label='Actual', color='black', linewidth=1.5)
ax.plot(
  x, pred_median[:n_samples], label='Predicted (median)', color='blue', linewidth=1
)
ax.fill_between(
  x,
  pred_q10[:n_samples],
  pred_q90[:n_samples],
  alpha=0.3,
  color='blue',
  label='80% prediction interval',
)

ax.set_xlabel('Sample')
ax.set_ylabel('next_close_log (normalized)')
ax.set_title('TFT Predictions vs Actuals (Test Set Sample)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()